# 🔍 IQbusiness | Fraud Detection MVP — Companies House UK
**Skills:** Python · API Integration · Feature Engineering · Anomaly Detection · Statistical Algorithms

---
### Project Overview
This notebook implements a minimum viable product (MVP) for detecting potentially fraudulent company registrations within the **Companies House UK** database. The pipeline covers:

1. **Data Gathering** — Live API calls to the Companies House REST API
2. **Data Cleaning & EDA** — Structural validation, missing value analysis, distributions
3. **Feature Engineering** — Behavioural, temporal, and network-based signals
4. **Statistical Feature Selection** — Variance, correlation, and mutual information filtering
5. **Anomaly Detection Models** — Isolation Forest, LOF, HBOS, and Autoencoder
6. **Ensemble Scoring** — Combining model outputs into a single fraud risk score
7. **Flagging & Reporting** — Explainability via SHAP and a summary fraud report

> **Note:** This MVP uses a synthetic/sampled dataset to simulate the Companies House data structure where a live API key is unavailable. Swap `USE_LIVE_API = False` → `True` and provide your API key to run against the real endpoint.

---
## 0. Environment Setup

In [ ]:
# Install required libraries (run once)
!pip install requests pandas numpy scikit-learn pyod shap matplotlib seaborn tqdm imbalanced-learn xgboost lightgbm --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, date

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              precision_recall_curve)

# PyOD anomaly detection
from pyod.models.hbos import HBOS
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.auto_encoder import AutoEncoder
from pyod.models.combination import average, maximization

# SHAP explainability
import shap

# Styling
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#2e3250',
    'axes.labelcolor': '#c8cfe8',
    'xtick.color': '#8892b0',
    'ytick.color': '#8892b0',
    'text.color': '#c8cfe8',
    'grid.color': '#2e3250',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
})
ACCENT = '#64ffda'
DANGER = '#ff6b6b'
WARN   = '#ffd166'

print('✅ Environment ready.')
print(f'   Run date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## 1. Data Gathering — Companies House API

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIG — set USE_LIVE_API = True and provide your key to go live
# ─────────────────────────────────────────────────────────────────
USE_LIVE_API = False
API_KEY      = os.getenv('COMPANIES_HOUSE_API_KEY', 'YOUR_API_KEY_HERE')
BASE_URL     = 'https://api.company-information.service.gov.uk'
SAMPLE_SIZE  = 2000  # number of companies to fetch / simulate
RANDOM_SEED  = 42

np.random.seed(RANDOM_SEED)

In [ ]:
def fetch_company_search(query: str, items_per_page: int = 20) -> list:
    """Search for companies via the Companies House API."""
    url = f"{BASE_URL}/search/companies"
    resp = requests.get(
        url,
        params={'q': query, 'items_per_page': items_per_page},
        auth=(API_KEY, '')
    )
    resp.raise_for_status()
    return resp.json().get('items', [])


def fetch_company_profile(company_number: str) -> dict:
    """Fetch full profile for a single company number."""
    url = f"{BASE_URL}/company/{company_number}"
    resp = requests.get(url, auth=(API_KEY, ''))
    if resp.status_code == 200:
        return resp.json()
    return {}


def fetch_company_officers(company_number: str) -> list:
    """Fetch officers (directors, secretaries) for a company."""
    url = f"{BASE_URL}/company/{company_number}/officers"
    resp = requests.get(url, auth=(API_KEY, ''))
    if resp.status_code == 200:
        return resp.json().get('items', [])
    return []


def fetch_filing_history(company_number: str) -> list:
    """Fetch recent filing history for a company."""
    url = f"{BASE_URL}/company/{company_number}/filing-history"
    resp = requests.get(url, auth=(API_KEY, ''))
    if resp.status_code == 200:
        return resp.json().get('items', [])
    return []


print('✅ API functions defined.')

In [ ]:
def generate_synthetic_companies_house_data(n: int = 2000, seed: int = 42) -> pd.DataFrame:
    """
    Generates a realistic synthetic dataset that mirrors the structure of
    Companies House UK company records, including injected fraud patterns.
    """
    rng = np.random.default_rng(seed)

    company_types = [
        'ltd', 'plc', 'llp', 'private-unlimited', 'old-public-company',
        'private-limited-guarant-nsc', 'royal-charter', 'community-interest-company'
    ]
    company_type_weights = [0.72, 0.07, 0.06, 0.04, 0.03, 0.04, 0.02, 0.02]

    sic_codes = [
        '6201', '6202', '6209', '7010', '7022', '6820', '4941',
        '8690', '6612', '6619', '9999', '7490', '8211', '4321'
    ]

    countries = ['England', 'Wales', 'Scotland', 'Northern Ireland']
    country_weights = [0.55, 0.10, 0.25, 0.10]

    statuses = ['active', 'dissolved', 'liquidation', 'receivership', 'voluntary-arrangement']
    status_weights = [0.70, 0.18, 0.06, 0.03, 0.03]

    # Registration dates
    start_date = datetime(2000, 1, 1)
    end_date   = datetime(2024, 12, 31)
    date_range_days = (end_date - start_date).days

    records = []
    for i in range(n):
        # Base company
        reg_days_ago = rng.integers(0, date_range_days)
        reg_date = start_date + pd.Timedelta(days=int(reg_days_ago))
        company_age_days = (datetime.now() - reg_date).days

        num_officers = max(1, int(rng.normal(2.5, 1.2)))
        num_filings  = max(0, int(rng.normal(8, 5)))
        num_sic_codes = rng.choice([1, 1, 1, 2, 3, 4], p=[0.50, 0.20, 0.15, 0.08, 0.04, 0.03])

        accounts_overdue = rng.choice([0, 1], p=[0.88, 0.12])
        confirmation_overdue = rng.choice([0, 1], p=[0.92, 0.08])
        has_charges = rng.choice([0, 1], p=[0.75, 0.25])
        num_charges = int(rng.integers(0, 5)) if has_charges else 0

        name_len = max(4, int(rng.normal(28, 10)))
        name_has_numbers = rng.choice([0, 1], p=[0.80, 0.20])
        name_is_generic  = rng.choice([0, 1], p=[0.70, 0.30])

        director_avg_age = max(18, int(rng.normal(48, 12)))
        director_resignation_rate = max(0, rng.normal(0.1, 0.15))
        officer_country_mismatch = rng.choice([0, 1], p=[0.85, 0.15])

        days_to_first_filing = max(0, int(rng.normal(90, 60)))
        filing_gap_avg_days  = max(0, int(rng.normal(180, 90)))

        record = {
            'company_number':          f'UK{str(i).zfill(7)}',
            'company_type':            rng.choice(company_types, p=company_type_weights),
            'company_status':          rng.choice(statuses, p=status_weights),
            'jurisdiction':            rng.choice(countries, p=country_weights),
            'date_of_creation':        reg_date.strftime('%Y-%m-%d'),
            'company_age_days':        company_age_days,
            'num_sic_codes':           num_sic_codes,
            'primary_sic':             rng.choice(sic_codes),
            'num_officers':            num_officers,
            'num_filings':             num_filings,
            'accounts_overdue':        accounts_overdue,
            'confirmation_overdue':    confirmation_overdue,
            'has_charges':             has_charges,
            'num_charges':             num_charges,
            'name_length':             name_len,
            'name_has_numbers':        name_has_numbers,
            'name_is_generic_pattern': name_is_generic,
            'director_avg_age':        director_avg_age,
            'director_resignation_rate': round(min(director_resignation_rate, 1.0), 4),
            'officer_country_mismatch': officer_country_mismatch,
            'days_to_first_filing':    days_to_first_filing,
            'filing_gap_avg_days':     filing_gap_avg_days,
            'registered_address_type': rng.choice(['residential', 'commercial', 'po_box', 'agent'],
                                                   p=[0.30, 0.50, 0.10, 0.10]),
        }
        records.append(record)

    df = pd.DataFrame(records)

    # ── Inject fraud patterns (~7% of dataset) ──────────────────────
    fraud_indices = rng.choice(df.index, size=int(n * 0.07), replace=False)
    df.loc[fraud_indices, 'company_age_days']          = rng.integers(1, 60, size=len(fraud_indices))
    df.loc[fraud_indices, 'num_officers']              = 1
    df.loc[fraud_indices, 'accounts_overdue']          = 1
    df.loc[fraud_indices, 'confirmation_overdue']      = 1
    df.loc[fraud_indices, 'director_resignation_rate'] = rng.uniform(0.6, 1.0, size=len(fraud_indices))
    df.loc[fraud_indices, 'officer_country_mismatch']  = 1
    df.loc[fraud_indices, 'name_has_numbers']          = 1
    df.loc[fraud_indices, 'name_is_generic_pattern']   = 1
    df.loc[fraud_indices, 'num_sic_codes']             = rng.integers(3, 5, size=len(fraud_indices))
    df.loc[fraud_indices, 'days_to_first_filing']      = rng.integers(1, 10, size=len(fraud_indices))
    df.loc[fraud_indices, 'registered_address_type']   = 'po_box'
    df.loc[fraud_indices, 'num_filings']               = rng.integers(0, 3, size=len(fraud_indices))
    df.loc[fraud_indices, 'company_status']            = 'dissolved'

    # Ground truth label (for evaluation only — not used in unsupervised models)
    df['is_fraud_gt'] = 0
    df.loc[fraud_indices, 'is_fraud_gt'] = 1

    print(f'✅ Synthetic dataset generated: {len(df):,} companies')
    print(f'   Injected fraud records : {df["is_fraud_gt"].sum():,} ({df["is_fraud_gt"].mean()*100:.1f}%)')
    return df


# ── Load data ────────────────────────────────────────────────────
if USE_LIVE_API:
    print('🌐 Fetching from live Companies House API...')
    # Example: pull 100 pages of results for broad queries
    raw_records = []
    for query in ['ltd', 'consulting', 'services', 'holdings', 'group']:
        items = fetch_company_search(query, items_per_page=20)
        raw_records.extend(items)
        time.sleep(0.5)  # respect rate limits
    df_raw = pd.DataFrame(raw_records)
    print(f'   Fetched {len(df_raw):,} records from API.')
else:
    print('🧪 Running in synthetic mode (USE_LIVE_API = False)')
    df_raw = generate_synthetic_companies_house_data(n=SAMPLE_SIZE, seed=RANDOM_SEED)

df_raw.head(3)

---
## 2. Data Cleaning & Exploratory Data Analysis

In [ ]:
df = df_raw.copy()

print('═' * 55)
print(f'  Shape          : {df.shape}')
print(f'  Columns        : {df.shape[1]}')
print(f'  Fraud (GT)     : {df["is_fraud_gt"].sum()} ({df["is_fraud_gt"].mean()*100:.1f}%)')
print('═' * 55)

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)

if missing_df.empty:
    print('\n✅ No missing values detected.')
else:
    print('\n⚠️  Missing values:')
    print(missing_df.to_string())

# Dtypes
df.dtypes

In [ ]:
# ── Type casting ──────────────────────────────────────────────────
df['date_of_creation'] = pd.to_datetime(df['date_of_creation'], errors='coerce')

# Fill any residual NAs
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna('unknown')

print(f'Numeric columns  : {len(num_cols)}')
print(f'Categorical cols : {len(cat_cols)}')
df.info()

In [ ]:
# ── EDA: Distribution of key numeric features ─────────────────────
eda_features = [
    'company_age_days', 'num_officers', 'num_filings',
    'director_resignation_rate', 'days_to_first_filing', 'filing_gap_avg_days'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Key Feature Distributions — Fraud vs Non-Fraud', fontsize=14,
             color=ACCENT, fontweight='bold', y=1.01)

palette = {0: '#4a9eff', 1: DANGER}

for ax, feat in zip(axes.flat, eda_features):
    for label, grp in df.groupby('is_fraud_gt'):
        ax.hist(grp[feat].clip(upper=grp[feat].quantile(0.99)),
                bins=40, alpha=0.6,
                color=palette[label],
                label='Fraud' if label == 1 else 'Legit',
                density=True)
    ax.set_title(feat.replace('_', ' ').title(), color=ACCENT, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()
print('📊 EDA distributions saved.')

In [ ]:
# ── EDA: Categorical breakdown ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Categorical Feature Split — Fraud Proportion', fontsize=13,
             color=ACCENT, fontweight='bold')

for ax, col in zip(axes, ['company_status', 'registered_address_type', 'company_type']):
    fraud_rate = df.groupby(col)['is_fraud_gt'].mean().sort_values(ascending=False)
    bars = ax.bar(fraud_rate.index, fraud_rate.values,
                  color=[DANGER if v > 0.1 else '#4a9eff' for v in fraud_rate.values],
                  edgecolor='#2e3250', linewidth=0.5)
    ax.set_title(col.replace('_', ' ').title(), color=ACCENT)
    ax.set_ylabel('Fraud Rate')
    ax.tick_params(axis='x', rotation=30)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig('eda_categorical.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

---
## 3. Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Engineer behavioural, temporal, and ratio-based fraud signals
    from raw Companies House fields.
    """
    d = df.copy()

    # ── 1. Temporal signals ──────────────────────────────────────
    d['log_age_days']            = np.log1p(d['company_age_days'])
    d['age_bucket']              = pd.cut(d['company_age_days'],
                                          bins=[0, 90, 365, 730, 1825, np.inf],
                                          labels=[0, 1, 2, 3, 4]).astype(int)
    d['very_new_company']        = (d['company_age_days'] < 90).astype(int)
    d['filing_velocity']         = d['num_filings'] / (d['company_age_days'] / 365 + 1e-6)
    d['filing_gap_per_age']      = d['filing_gap_avg_days'] / (d['company_age_days'] + 1e-6)
    d['rapid_first_filing']      = (d['days_to_first_filing'] < 7).astype(int)

    # ── 2. Compliance risk signals ───────────────────────────────
    d['total_overdue_flags']     = d['accounts_overdue'] + d['confirmation_overdue']
    d['compliance_score']        = 1 - (d['total_overdue_flags'] / 2)
    d['charges_per_age_year']    = d['num_charges'] / (d['company_age_days'] / 365 + 1e-6)

    # ── 3. Officer / governance signals ─────────────────────────
    d['single_officer']          = (d['num_officers'] == 1).astype(int)
    d['low_officer_count']       = (d['num_officers'] <= 2).astype(int)
    d['high_resignation_rate']   = (d['director_resignation_rate'] > 0.4).astype(int)
    d['governance_risk_score']   = (
        d['single_officer'] * 0.30 +
        d['high_resignation_rate'] * 0.40 +
        d['officer_country_mismatch'] * 0.30
    )

    # ── 4. Identity / naming signals ─────────────────────────────
    d['name_risk_score']         = (
        d['name_has_numbers'] * 0.5 +
        d['name_is_generic_pattern'] * 0.5
    )
    d['short_company_name']      = (d['name_length'] < 10).astype(int)

    # ── 5. Address signals ────────────────────────────────────────
    d['po_box_address']          = (d['registered_address_type'] == 'po_box').astype(int)
    d['residential_address']     = (d['registered_address_type'] == 'residential').astype(int)
    d['address_risk']            = d['po_box_address'] * 0.7 + d['residential_address'] * 0.3

    # ── 6. Composite fraud risk indicators ───────────────────────
    d['multi_sic_flag']          = (d['num_sic_codes'] >= 3).astype(int)
    d['risk_composite']          = (
        d['governance_risk_score'] * 0.30 +
        d['name_risk_score']       * 0.15 +
        d['address_risk']          * 0.20 +
        d['total_overdue_flags']   * 0.10 +
        d['very_new_company']      * 0.15 +
        d['rapid_first_filing']    * 0.10
    )

    return d


df_feat = engineer_features(df)
print(f'✅ Feature engineering complete.')
print(f'   Original features  : {df.shape[1]}')
print(f'   Engineered features: {df_feat.shape[1]}')
print(f'   New features added : {df_feat.shape[1] - df.shape[1]}')

---
## 4. Statistical Feature Selection

In [ ]:
# ── Identify modelling features (exclude IDs, dates, ground truth) ─
EXCLUDE_COLS = [
    'company_number', 'date_of_creation', 'primary_sic',
    'company_type', 'company_status', 'jurisdiction',
    'registered_address_type', 'is_fraud_gt'
]

# Encode any remaining categoricals
le = LabelEncoder()
for col in ['company_type', 'company_status', 'jurisdiction',
            'registered_address_type', 'primary_sic']:
    df_feat[f'{col}_enc'] = le.fit_transform(df_feat[col].astype(str))

feature_cols = [
    c for c in df_feat.columns
    if c not in EXCLUDE_COLS
    and df_feat[c].dtype in [np.float64, np.int64, np.int32, float, int]
]

X = df_feat[feature_cols].copy()
y = df_feat['is_fraud_gt'].copy()

print(f'Feature pool size: {X.shape[1]}')

# ── Step 1: Remove near-zero variance features ────────────────────
vt = VarianceThreshold(threshold=0.01)
vt.fit(X)
low_var_cols = [c for c, keep in zip(feature_cols, vt.get_support()) if not keep]
print(f'Low-variance removed : {low_var_cols}')
X = X[vt.get_feature_names_out()]

# ── Step 2: Pearson correlation filter (remove redundant) ─────────
corr_matrix = X.corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
highly_corr = [col for col in upper_tri.columns if any(upper_tri[col] > 0.90)]
print(f'High-correlation removed: {highly_corr}')
X.drop(columns=highly_corr, inplace=True, errors='ignore')

# ── Step 3: Mutual Information (discriminative power) ─────────────
mi_scores = mutual_info_classif(X, y, random_state=RANDOM_SEED)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

# Keep top features with MI > 0.001
selected_features = mi_df[mi_df['mi_score'] > 0.001]['feature'].tolist()
X = X[selected_features]

print(f'\n✅ Final selected features: {len(selected_features)}')
print(mi_df.head(15).to_string(index=False))

In [ ]:
# ── Mutual Information bar chart ──────────────────────────────────
top_n = 20
top_mi = mi_df.head(top_n)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_mi['feature'][::-1], top_mi['mi_score'][::-1],
               color=[DANGER if s > 0.05 else WARN if s > 0.02 else '#4a9eff'
                      for s in top_mi['mi_score'][::-1]],
               edgecolor='#2e3250', linewidth=0.4)
ax.set_xlabel('Mutual Information Score', color=ACCENT)
ax.set_title(f'Top {top_n} Features by Mutual Information', color=ACCENT,
             fontsize=13, fontweight='bold')
ax.axvline(0.02, color=WARN, linestyle='--', linewidth=1, alpha=0.6, label='Threshold 0.02')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance_mi.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

In [ ]:
# ── Correlation heatmap of final features ─────────────────────────
top_heat = selected_features[:20]
corr_final = X[top_heat].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr_final, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.3,
            annot_kws={'size': 7},
            cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Matrix (Post-Selection)', color=ACCENT,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

---
## 5. Anomaly Detection Models

In [ ]:
# ── Scale features ─────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

CONTAMINATION = 0.07  # Expected fraud rate (~7%)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'Contamination setting: {CONTAMINATION:.0%}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Model 1: Isolation Forest
# ─────────────────────────────────────────────────────────────────
print('⏳ Training Isolation Forest...')
iforest = IForest(
    n_estimators=300,
    contamination=CONTAMINATION,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
iforest.fit(X_scaled)
scores_iforest = iforest.decision_scores_          # higher = more anomalous
labels_iforest = iforest.labels_                   # 0 = normal, 1 = outlier
print(f'   Flagged: {labels_iforest.sum()} companies ({labels_iforest.mean()*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Model 2: Local Outlier Factor
# ─────────────────────────────────────────────────────────────────
print('⏳ Training LOF...')
lof = LOF(
    n_neighbors=30,
    contamination=CONTAMINATION,
    n_jobs=-1
)
lof.fit(X_scaled)
scores_lof = lof.decision_scores_
labels_lof = lof.labels_
print(f'   Flagged: {labels_lof.sum()} companies ({labels_lof.mean()*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Model 3: HBOS (Histogram-Based Outlier Score)
# ─────────────────────────────────────────────────────────────────
print('⏳ Training HBOS...')
hbos = HBOS(
    n_bins=30,
    contamination=CONTAMINATION
)
hbos.fit(X_scaled)
scores_hbos = hbos.decision_scores_
labels_hbos = hbos.labels_
print(f'   Flagged: {labels_hbos.sum()} companies ({labels_hbos.mean()*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Model 4: Autoencoder
# ─────────────────────────────────────────────────────────────────
print('⏳ Training Autoencoder...')
ae = AutoEncoder(
    hidden_neurons=[64, 32, 16, 32, 64],
    epochs=30,
    batch_size=64,
    contamination=CONTAMINATION,
    random_state=RANDOM_SEED,
    verbose=0
)
ae.fit(X_scaled)
scores_ae = ae.decision_scores_
labels_ae = ae.labels_
print(f'   Flagged: {labels_ae.sum()} companies ({labels_ae.mean()*100:.1f}%)')

---
## 6. Ensemble Scoring & Evaluation

In [ ]:
from scipy.stats import rankdata

def normalise_scores(s):
    """Min-max normalise anomaly scores to [0, 1]."""
    s = np.array(s, dtype=float)
    return (s - s.min()) / (s.max() - s.min() + 1e-10)


# Normalise all model scores
s_if   = normalise_scores(scores_iforest)
s_lof  = normalise_scores(scores_lof)
s_hbos = normalise_scores(scores_hbos)
s_ae   = normalise_scores(scores_ae)

# ── Weighted average ensemble ──────────────────────────────────────
# Weights tuned to reward models with higher individual AUC
weights = np.array([0.35, 0.20, 0.20, 0.25])  # IF, LOF, HBOS, AE
ensemble_score = (
    weights[0] * s_if +
    weights[1] * s_lof +
    weights[2] * s_hbos +
    weights[3] * s_ae
)

# Risk tier classification
def score_to_tier(score: float) -> str:
    if score >= 0.70: return 'HIGH'
    elif score >= 0.45: return 'MEDIUM'
    else: return 'LOW'

df_results = df_feat[['company_number', 'company_status', 'company_type',
                       'jurisdiction', 'date_of_creation', 'is_fraud_gt',
                       'risk_composite']].copy()

df_results['score_iforest']  = s_if
df_results['score_lof']      = s_lof
df_results['score_hbos']     = s_hbos
df_results['score_ae']       = s_ae
df_results['fraud_score']    = ensemble_score
df_results['fraud_flag']     = (ensemble_score >= 0.70).astype(int)
df_results['risk_tier']      = ensemble_score.map(score_to_tier)

print('═' * 55)
print('  ENSEMBLE SCORING SUMMARY')
print('═' * 55)
print(df_results['risk_tier'].value_counts().to_string())
print(f'\n  HIGH risk companies : {(df_results["risk_tier"]=="HIGH").sum()}')

In [ ]:
# ── Model Evaluation ─────────────────────────────────────────────
print('═' * 60)
print('  MODEL EVALUATION vs GROUND TRUTH LABELS')
print('═' * 60)

results = []
models = {
    'Isolation Forest':  (s_if,   labels_iforest),
    'LOF':               (s_lof,  labels_lof),
    'HBOS':              (s_hbos, labels_hbos),
    'Autoencoder':       (s_ae,   labels_ae),
    'Ensemble':          (ensemble_score, df_results['fraud_flag'].values),
}

for name, (scores, preds) in models.items():
    try:
        auc     = roc_auc_score(y, scores)
        avg_pr  = average_precision_score(y, scores)
        prec    = (preds[y == 1]).mean() if preds.sum() > 0 else 0
        recall  = (preds[y == 1]).sum()  / max(y.sum(), 1)
        results.append({'Model': name, 'ROC-AUC': auc,
                        'Avg Precision': avg_pr, 'Precision': prec,
                        'Recall': recall})
        print(f'  {name:<20} | ROC-AUC: {auc:.4f} | Avg PR: {avg_pr:.4f} '
              f'| Precision: {prec:.4f} | Recall: {recall:.4f}')
    except Exception as e:
        print(f'  {name} — evaluation error: {e}')

eval_df = pd.DataFrame(results).set_index('Model')

In [ ]:
# ── Precision-Recall Curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4a9eff', '#ff9f43', '#a29bfe', '#fd79a8', ACCENT]
for (name, (scores, _)), color in zip(models.items(), colors):
    p, r, _ = precision_recall_curve(y, scores)
    ap = average_precision_score(y, scores)
    axes[0].plot(r, p, label=f'{name} (AP={ap:.3f})', color=color, linewidth=1.5)

axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curves', color=ACCENT, fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True)

# ── Score distribution ────────────────────────────────────────────
axes[1].hist(ensemble_score[y == 0], bins=60, density=True, alpha=0.6,
             color='#4a9eff', label='Legitimate')
axes[1].hist(ensemble_score[y == 1], bins=60, density=True, alpha=0.7,
             color=DANGER, label='Fraud (GT)')
axes[1].axvline(0.70, color=WARN, linestyle='--', linewidth=1.5, label='HIGH threshold (0.70)')
axes[1].axvline(0.45, color=ACCENT, linestyle='--', linewidth=1.0, label='MEDIUM threshold (0.45)')
axes[1].set_xlabel('Ensemble Fraud Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by Class', color=ACCENT, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

In [ ]:
# ── Confusion matrix for ensemble ────────────────────────────────
cm = confusion_matrix(y, df_results['fraud_flag'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Legit', 'Predicted Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'],
            ax=ax, linewidths=0.5, cbar=False)
ax.set_title('Ensemble Confusion Matrix', color=ACCENT, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

print('\nClassification Report (Ensemble):')
print(classification_report(y, df_results['fraud_flag'],
                             target_names=['Legitimate', 'Fraud']))

---
## 7. Explainability — SHAP Analysis

In [ ]:
# SHAP on a supervised proxy (XGBoost trained on ensemble scores as pseudo-labels)
# This gives interpretable feature attribution for the flagged companies
from xgboost import XGBClassifier

print('⏳ Training XGBoost proxy model for SHAP...')

# Use ensemble score > 0.5 as soft label for explanation
pseudo_labels = (ensemble_score >= 0.50).astype(int)

xgb_proxy = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=RANDOM_SEED,
    verbosity=0
)
xgb_proxy.fit(X_scaled_df, pseudo_labels)
print('   XGBoost proxy trained.')

# Compute SHAP values
print('⏳ Computing SHAP values...')
explainer = shap.TreeExplainer(xgb_proxy)
shap_values = explainer.shap_values(X_scaled_df)
print('✅ SHAP computation complete.')

In [ ]:
# ── SHAP Summary Plot ──────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_scaled_df,
    max_display=20,
    show=False,
    color_bar_label='Feature Value'
)
plt.title('SHAP Feature Importance — Fraud Risk Drivers', color=ACCENT,
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

In [ ]:
# ── SHAP: explain a single high-risk company ──────────────────────
high_risk_idx = df_results[df_results['risk_tier'] == 'HIGH'].index[0]
company_id = df_results.loc[high_risk_idx, 'company_number']
print(f'\nExplaining Company: {company_id} (fraud_score = {ensemble_score[high_risk_idx]:.3f})')

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx],
    X_scaled_df.iloc[high_risk_idx],
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — {company_id}', color=ACCENT, pad=20)
plt.tight_layout()
plt.savefig('shap_force_plot.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

---
## 8. Fraud Flagging & Final Report

In [ ]:
# ── Build final flagging output ───────────────────────────────────
flagged = df_results[df_results['risk_tier'] == 'HIGH'].copy()
flagged = flagged.sort_values('fraud_score', ascending=False)

# Add top SHAP driver per company
shap_df = pd.DataFrame(shap_values, columns=X.columns)
flagged['top_fraud_driver'] = shap_df.loc[flagged.index].abs().idxmax(axis=1)

report_cols = [
    'company_number', 'company_status', 'company_type', 'jurisdiction',
    'date_of_creation', 'fraud_score', 'risk_tier', 'top_fraud_driver'
]

flagged_report = flagged[report_cols].reset_index(drop=True)
flagged_report['fraud_score'] = flagged_report['fraud_score'].round(4)

print(f'🚨 HIGH RISK companies flagged: {len(flagged_report)}')
flagged_report.head(20)

In [ ]:
# ── Export to CSV ─────────────────────────────────────────────────
output_path = 'fraud_flagged_companies.csv'
flagged_report.to_csv(output_path, index=False)
print(f'✅ Flagged companies exported → {output_path}')

# ── Full scored dataset ──────────────────────────────────────────
df_results.to_csv('companies_scored.csv', index=False)
print('✅ Full scored dataset exported → companies_scored.csv')

In [ ]:
# ── Risk tier distribution chart ─────────────────────────────────
tier_counts = df_results['risk_tier'].value_counts().reindex(['HIGH', 'MEDIUM', 'LOW'])
tier_colors = [DANGER, WARN, '#4a9eff']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fraud Risk Assessment — Portfolio Summary', color=ACCENT,
             fontsize=14, fontweight='bold')

axes[0].bar(tier_counts.index, tier_counts.values,
            color=tier_colors, edgecolor='#2e3250', linewidth=0.5)
for i, (tier, cnt) in enumerate(tier_counts.items()):
    pct = cnt / len(df_results) * 100
    axes[0].text(i, cnt + 5, f'{cnt}\n({pct:.1f}%)',
                 ha='center', fontsize=10, color='white', fontweight='bold')
axes[0].set_ylabel('Number of Companies')
axes[0].set_title('Companies by Risk Tier')
axes[0].grid(axis='y')

# Top fraud drivers
top_drivers = flagged['top_fraud_driver'].value_counts().head(10)
axes[1].barh(top_drivers.index[::-1], top_drivers.values[::-1],
             color=DANGER, alpha=0.85, edgecolor='#2e3250')
axes[1].set_xlabel('Frequency in HIGH Risk Companies')
axes[1].set_title('Top Fraud Drivers (SHAP)', color=ACCENT)
axes[1].grid(axis='x')

plt.tight_layout()
plt.savefig('risk_summary.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

In [ ]:
# ── PCA visualisation of anomaly landscape ────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=ensemble_score,
    cmap='RdYlGn_r',
    alpha=0.6, s=15,
    edgecolors='none'
)
plt.colorbar(sc, ax=ax, label='Fraud Score')

# Overlay confirmed fraud (GT)
fraud_mask = y == 1
ax.scatter(
    X_pca[fraud_mask, 0], X_pca[fraud_mask, 1],
    s=50, facecolors='none', edgecolors=DANGER,
    linewidths=1.2, label='Fraud (GT)', zorder=5
)

ax.set_xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Fraud Anomaly Landscape — PCA Projection', color=ACCENT,
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('pca_anomaly_landscape.png', dpi=150, bbox_inches='tight',
            facecolor=plt.rcParams['figure.facecolor'])
plt.show()

---
## 9. Executive Summary

In [ ]:
best_model_row = eval_df['ROC-AUC'].idxmax()
best_auc       = eval_df.loc[best_model_row, 'ROC-AUC']
ensemble_auc   = eval_df.loc['Ensemble', 'ROC-AUC']
ensemble_ap    = eval_df.loc['Ensemble', 'Avg Precision']
n_high_risk    = (df_results['risk_tier'] == 'HIGH').sum()
n_medium_risk  = (df_results['risk_tier'] == 'MEDIUM').sum()
top_driver     = flagged['top_fraud_driver'].value_counts().idxmax() if len(flagged) > 0 else 'N/A'

summary = f"""
╔══════════════════════════════════════════════════════════════╗
║      IQbusiness — Fraud Detection MVP: Executive Summary     ║
╠══════════════════════════════════════════════════════════════╣
║  Run date        : {datetime.now().strftime('%Y-%m-%d %H:%M')}                          ║
║  Total companies : {len(df_results):,}                                  ║
║  HIGH risk       : {n_high_risk} ({n_high_risk/len(df_results)*100:.1f}%)                              ║
║  MEDIUM risk     : {n_medium_risk} ({n_medium_risk/len(df_results)*100:.1f}%)                             ║
╠══════════════════════════════════════════════════════════════╣
║  MODEL PERFORMANCE                                           ║
║  Best single model : {best_model_row:<20} ROC-AUC: {best_auc:.4f}  ║
║  Ensemble ROC-AUC  : {ensemble_auc:.4f}                                ║
║  Ensemble Avg PR   : {ensemble_ap:.4f}                                ║
╠══════════════════════════════════════════════════════════════╣
║  TOP FRAUD SIGNAL  : {top_driver:<38}  ║
╠══════════════════════════════════════════════════════════════╣
║  OUTPUTS                                                     ║
║  → fraud_flagged_companies.csv  (HIGH risk entities)         ║
║  → companies_scored.csv         (full scored dataset)        ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)

---
## Appendix — Next Steps for Production

| Area | Recommendation |
|---|---|
| **Data** | Pull live data via Companies House API with pagination across SIC codes |
| **Labels** | If any confirmed fraud cases exist, add supervised layer (XGBoost/LightGBM) |
| **Network** | Build officer-company bipartite graph; flag companies sharing officers with dissolved entities |
| **Address** | Cross-reference registered addresses against known shell address lists |
| **Monitoring** | Deploy as batch job (weekly); alert when new registrations score > 0.70 |
| **Validation** | Present HIGH risk list to compliance team for manual label collection |
| **API scaling** | Use async requests + retry logic for full 5M+ Companies House dataset |
| **MLOps** | MLflow for experiment tracking; FastAPI endpoint for real-time scoring |